# Préparation des données SFT — MediQAl (mcqm)

Complément français du SFT : QCM à **réponses multiples** (configuration `mcqm` de MediQAl, licence CC-BY-4.0). La réponse attendue est la **liste des bonnes propositions**, lettre et texte (« - A. … »).

- **Pourquoi** : le SFT `mcqu` n'apprend que des réponses uniques. Le DPO français (`data_prep_DPO_MediQAl_mcqm`) compare des listes : le modèle doit avoir appris ce format avant le DPO, qui ajuste des préférences mais n'enseigne pas un nouveau format.
- **Deux parties** :
  - questions **sans cas clinique** (connaissances générales), découpage officiel de MediQAl ;
  - questions **avec cas clinique**, sur les cas réservés au SFT (`projet_14.mediqal`). Sans elles, le modèle associerait « cas clinique » à « une seule réponse » (le seul format vu avec un cas dans `mcqu`).
- **Pas de recouvrement avec le DPO** : chaque cas clinique va soit au SFT, soit au DPO, et dans un seul split. Un contrôle final le vérifie sur tous les fichiers MediQAl.
- **Sortie** : `data/anonymized/sft_mediqal_mcqm_{train,validation,test}.anonymized.jsonl`, au format `messages` du SFT.
- **Ordre d'exécution** : après `data_prep_MediQAl` et `data_prep_DPO_MediQAl_mcqm` (contrôle final).

# Import

In [1]:
import json
import re

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

from projet_14.mediqal import (
    OPTION_COLUMNS, OPTIONS_HEADER, SYSTEM_PROMPT, assign_cases, clean_text, correct_letters, format_answer, user_content,
)
from projet_14.rgpd import run

SEED = 42

# Taille maximale des jeux de questions sans cas clinique (les questions avec cas sont toutes gardées)
N_TRAIN = 1000
N_VALIDATION = 200
N_TEST = 500

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqm")
df = pd.concat([ds[name].to_pandas().assign(official_split=name) for name in ds], ignore_index=True)

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type,official_split
0,11399,Plusieurs jours après un contage avec un enfan...,Quelle(s) proposition(s) peut (peuvent) s'appl...,Transmission manuportée,La phase de contagiosité dure 8 jours,Maladie strictement humaine,Confère une immunité,Transmission indirecte possible,"C,D",QCM,Infectious Diseases,Understanding,train
1,17928,NaN,Quel(s) germe(s) est (sont) responsable(s) de ...,Gardnerella Vaginalis,Trichomonas,Tréponèmes,Candida albicans,Strepto B,"A,B",QCM,Gynecology and Obstetrics,Understanding,train
2,17520,"Une jeune femme de 26 ans, d'origine française...","En cours d'hospitalisation, laquelle ou lesque...",Réduction de l'apport en protéines animales,Association d'une sulfonylurée à l'insuline,Conservation du même type d'insuline mais augm...,Education à l'autosurveillance glycémique,"Adaptation d'un nouveau schéma thérapeutique, ...","A,D,E",QCM,Endocrinology and Metabolism,Reasoning,train
3,10875,NaN,Quelles sont parmi les bactéries suivantes res...,Staphylococcus aureus,Salmonella enteritidis,Shigella sonnei,Vibrio cholerae,Campylobacter fetus,"B,C,E",QCM,Pediatrics,Understanding,train
4,14691,NaN,"Parmi les 5 propositions suivantes, quelle est...",Il est plus souvent d'origine maternelle,La majorité des formes sont inapparentes,Une primo-infection génitale maternelle durant...,L'accouchement par césarienne est indiqué lors...,Aucune chimiothérapie antivirale n'est active,"A,C,D",QCM,Pediatrics,Understanding,train


In [4]:
# Questions avec et sans cas clinique, par split officiel
pd.crosstab(df["official_split"], df["clinical_case"].notna().map({True: "avec cas", False: "sans cas"}))

clinical_case,avec cas,sans cas
official_split,,
test,1342,2042
train,2185,3582
validation,547,919


In [5]:
# Nombre de bonnes réponses par question
df["correct_answers"].str.split(",").map(len).value_counts().sort_index().rename_axis("bonnes réponses").to_frame("questions")

,questions
bonnes réponses,
2,4352
3,3802
4,1885
5,578


# Nettoyage

In [6]:
df = df.assign(**{col: df[col].map(clean_text, na_action="ignore") for col in ["clinical_case", "question"] + OPTION_COLUMNS})
n_before = len(df)
df = df.drop_duplicates(subset=["clinical_case", "question"] + OPTION_COLUMNS)
print(f"{n_before} rows, {len(df)} après dédoublonnage")

10617 rows, 10501 après dédoublonnage


# Questions sans cas clinique

Pas de cas à répartir : on garde le découpage officiel. Une même question peut toutefois apparaître dans deux splits : on la retire du train si elle est aussi en validation ou en test.

In [7]:
no_case = df[df["clinical_case"].isna()]
no_case_splits = {name: no_case[no_case["official_split"] == name] for name in ["train", "validation", "test"]}

held_out = pd.concat([no_case_splits["validation"]["question"], no_case_splits["test"]["question"]])
no_case_splits["train"] = no_case_splits["train"][~no_case_splits["train"]["question"].isin(held_out)]
no_case_splits["validation"] = no_case_splits["validation"][
    ~no_case_splits["validation"]["question"].isin(no_case_splits["test"]["question"])
]

for name, size in {"train": N_TRAIN, "validation": N_VALIDATION, "test": N_TEST}.items():
    n_before = len(no_case_splits[name])
    no_case_splits[name] = no_case_splits[name].sample(n=min(size, n_before), random_state=SEED)
    print(f"{name}: {n_before} questions sans cas, {len(no_case_splits[name])} gardées")

train: 3398 questions sans cas, 1000 gardées
validation: 876 questions sans cas, 200 gardées
test: 2004 questions sans cas, 500 gardées


# Questions avec cas clinique

Même répartition par cas que dans les autres notebooks MediQAl (`assign_cases`) : on garde les cas réservés au SFT, dans leur split.

In [8]:
with_case = assign_cases(df.dropna(subset=["clinical_case"]))
with_case = with_case[with_case["usage"] == "sft"]
case_splits = {name: with_case[with_case["split"] == name] for name in ["train", "validation", "test"]}
for name, d in case_splits.items():
    print(f"{name}: {len(d)} questions avec cas, {d['case_id'].nunique()} cas")

train: 1400 questions avec cas, 532 cas
validation: 180 questions avec cas, 70 cas
test: 431 questions avec cas, 167 cas


In [9]:
splits = {name: pd.concat([no_case_splits[name], case_splits[name]]) for name in ["train", "validation", "test"]}
pd.DataFrame({
    name: d["clinical_case"].notna().map({True: "avec cas", False: "sans cas"}).value_counts() for name, d in splits.items()
})

,train,validation,test
clinical_case,,,
avec cas,1400,180,431
sans cas,1000,200,500


# Réponse attendue

Liste à puces des bonnes propositions, lettre et texte, dans l'ordre A à E : **exactement le format** des réponses `chosen` du DPO français. Les propositions sont dans le prompt : le modèle choisit parmi elles, et l'exactitude se mesure sur les lettres.

In [10]:
for name, d in splits.items():
    splits[name] = d.assign(answer=d.apply(lambda row: format_answer(row, correct_letters(row), as_list=True), axis=1))

example = splits["train"][splits["train"]["clinical_case"].notna()].iloc[0]
print(f"{user_content(example)[-600:]}\n\nRÉPONSE :\n{example['answer']}")

sieurs jours après un contage avec un enfant atteint d'oreillons, une fillette de 6 ans développe un état fébrile à 40° rapidement suivi d'une parotidite unilatérale. En 48h, l'atteinte salivaire se bilatéralise et la température se normalise. Vers le 8e jour cependant, apparaissent des céphalées intenses et des vomissements incoercibles

Quelle(s) proposition(s) peut (peuvent) s'appliquer à l'épidémiologie de la maladie ?

Propositions :
A. Transmission manuportée
B. La phase de contagiosité dure 8 jours
C. Maladie strictement humaine
D. Confère une immunité
E. Transmission indirecte possible

RÉPONSE :
- C. Maladie strictement humaine
- D. Confère une immunité


# Build SFT JSONL file from dataset

In [11]:
def to_messages(row) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content(row)},  # cas clinique (s'il y en a un), question, propositions
        {"role": "assistant", "content": row["answer"]},
    ]


METADATA = ["id", "source", "case_id", "medical_subject", "question_type", "golden_answer", "lang"]

# Un fichier JSONL par split : ../data/sft_mediqal_mcqm_{train,validation,test}.jsonl
sft_ds = DatasetDict()
for name, d in splits.items():
    d = d.sample(frac=1, random_state=SEED).assign(
        source="MediQAl-mcqm", lang="fr", golden_answer=d["correct_answers"], messages=d.apply(to_messages, axis=1)
    )
    sft_ds[name] = Dataset.from_pandas(d[METADATA + ["messages"]], preserve_index=False)
    sft_ds[name].to_json(f"../data/sft_mediqal_mcqm_{name}.jsonl", force_ascii=False)

sft_ds["train"][0]["messages"]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 3/3 [00:00<00:00, 56.63ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 96.15ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 51.48ba/s]

[{'role': 'system',
  'content': 'Tu es un assistant médical expert. Réponds de façon précise et concise.'},
 {'role': 'user',
  'content': "Mlle B. , 28 ans, vient consulter pour demander une contraception oestroprogestative .Antécédents personnels : rien à signaler ; antécédents familiaux : un diabète non insulino-dépendant découvert chez sa mère l'année dernière. A l'examen : elle pèse 75 kgs pour 1,66 m ;  on note une certaine hypertrichose diffuse ; TA 15/9. L'examen est par ailleurs négatif. L'enquête alimentaire révèle un apport quotidien de 2 800 kcalories ainsi réparties : 100 g de protides - 110 g de lipides (surtout des graisses d'origine animale) - 300 g de glucides dont au moins 120 g apportés par des boissons très sucrées - 30 g d'alcool pur, essentiellement sous forme d'apéritifs. Un examen de sang récent a donné les résultats suivants : Cholestérol 5 mmol/l (1,90 g/l) - triglycérides 4 mmol/l (3,5 g/l) sérum lactescent et demeurant lactescent après 24 heures - glycémie 

In [12]:
print(sft_ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages'],
        num_rows: 2400
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages'],
        num_rows: 380
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages'],
        num_rows: 931
    })
})


# Anonymisation des données

On n'anonymise que le **cas clinique et la question** : message user (`roles=["user"]`), jusqu'à l'en-tête des propositions (`keep_after`). Les propositions et la réponse sont des énoncés rédigés par les enseignants, sans donnée patient : Presidio y prend des termes médicaux pour des noms (« Angiographie », « Érythrose palmaire »…) et effacerait ce que le modèle doit apprendre.

In [13]:
# Anonymisation de chaque split : ../data/anonymized/sft_mediqal_mcqm_{split}.anonymized.jsonl
reports = {
    name: run(
        f"../data/sft_mediqal_mcqm_{name}.jsonl", lang="fr", output_dir="../data/anonymized",
        roles=["user"], keep_after=OPTIONS_HEADER,
    )
    for name in sft_ds
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,2400,488,825
validation,380,58,115
test,931,176,309


In [14]:
# DataFrame des messages anonymisés (hors system) et contexte de chaque balise <PERSON>, pour vérification
rows = []
for name in sft_ds:
    with open(f"../data/anonymized/sft_mediqal_mcqm_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for message in record["messages"]:
                if message["role"] != "system":
                    rows.append({"split": name, "id": record["id"], "role": message["role"], "content": message["content"]})
contents_df = pd.DataFrame(rows)


def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


occurrences_df = contents_df[["split", "id", "role"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(min(20, len(occurrences_df)), random_state=SEED)

1043 occurrences de <PERSON>


,split,id,role,context
7386,test,18049,user,les sont du 12 Mars 1983 L'étude des antécédents de Madame <PERSON> vous permet de dire qu'elle appartient à une population à r
1826,train,16807,user,"Monsieur <PERSON> , 80 ans veuf, vivant seul est traité depuis un an pour une"
3588,train,18798,user,"Les 2 enfants de Mme <PERSON>.., 72 ans réfugié hongroise, viennent vous consulter pour o"
5772,test,12885,user,"M. <PERSON>, 58 ans. agriculteur, vient consulter pour une toux récidiv"
4416,train,13419,user,"<PERSON>, 50 ans, est hospitalisé pour une tentative de suicide médi"
7004,test,17901,user,"Madame <PERSON>. , 22 ans, vous consulte pour une lésion vulvaire douloureu"
3740,train,3260,user,"ses thérapeutiques """"coagulantes"""" (Reptilase, vitamine K1, <PERSON>) n'ont pu arrêter. Elle est hospitalisée avec un état d'ané"
1554,train,11635,user,ntes. Le médecin traitant a posé le diagnostic de rougeole <PERSON>) complication(s) peut-on craindre à court terme chez cet en
5792,test,21157,user,ivité. Sa mère s'est étonnée de la survenue d'une énurésie. <PERSON> a pourtant été propre de nuit comme de jour à l'âge de 4 an
3656,train,20970,user,"Mme <PERSON>.., 56 ans, est conduite au C.H.U. par le S.A.M.U. dans un t"


# Contrôle final

Sur **tous** les fichiers MediQAl (SFT `mcqu`, SFT `mcqm`, DPO `mcqm`) :

- aucun cas clinique partagé entre SFT et DPO ;
- aucun cas clinique partagé entre entraînement (train) et évaluation (validation, test), ni entre validation et test ;
- aucune question commune entre le SFT `mcqm` et le DPO.

In [15]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement
DATA_FILES = {name: f"../data/anonymized/sft_mediqal_mcqm_{name}.anonymized.jsonl" for name in sft_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

# Cas cliniques de chaque fichier MediQAl, par usage et par split
FILES = {"sft": ["sft_mediqal_mcqu", "sft_mediqal_mcqm"], "dpo": ["dpo_mediqal_mcqm"]}
cases, ids = {}, {"sft": set(), "dpo": set()}
for usage, stems in FILES.items():
    for stem in stems:
        for name in ["train", "validation", "test"]:
            with open(f"../data/anonymized/{stem}_{name}.anonymized.jsonl", encoding="utf-8") as f:
                records = [json.loads(line) for line in f]
            cases.setdefault((usage, name), set()).update(r["case_id"] for r in records if r["case_id"])
            if stem.endswith("mcqm"):
                ids[usage] |= {r["id"] for r in records}

sft_cases = set().union(*(c for (usage, _), c in cases.items() if usage == "sft"))
dpo_cases = set().union(*(c for (usage, _), c in cases.items() if usage == "dpo"))
assert not sft_cases & dpo_cases, f"{len(sft_cases & dpo_cases)} cas partagés entre SFT et DPO"
assert not ids["sft"] & ids["dpo"], f"{len(ids['sft'] & ids['dpo'])} questions communes entre SFT et DPO"
for usage in FILES:
    train, validation, test = (cases[(usage, name)] for name in ["train", "validation", "test"])
    assert not train & (validation | test), f"{usage} : cas partagé entre train et évaluation"
    assert not validation & test, f"{usage} : cas partagé entre validation et test"
print(f"OK : {len(sft_cases)} cas SFT et {len(dpo_cases)} cas DPO, sans recouvrement entre usages ni entre splits")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 2400 examples [00:00, 131544.74 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 380 examples [00:00, 51633.91 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 931 examples [00:00, 71412.32 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages', 'persons_replaced'],
        num_rows: 2400
    })
    validation: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages', 'persons_replaced'],
        num_rows: 380
    })
    test: Dataset({
        features: ['id', 'source', 'case_id', 'medical_subject', 'question_type', 'golden_answer', 'lang', 'messages', 'persons_replaced'],
        num_rows: 931
    })
})
OK : 1039 cas SFT et 767 cas DPO, sans recouvrement entre usages ni entre splits


# Limites

- **Questions sans cas clinique** : connaissances générales (« Parmi les propositions suivantes, concernant la grippe… »), moins proches du triage que les cas cliniques.
- **Réponses courtes, sans justification** : le modèle apprend *quelles* propositions retenir, pas à *expliquer* son choix. Les explications viennent de la partie anglaise (UltraMedical).